## Recall on Real World Dataset
1. Import real world matches with spectrum identifier and CID, pubchemlite to match CIDs to smiles, result smiles

2. How many matches, have a results, filter!(look up identifier)

3. Optional: Test weather binning method opposes the ground truth labeling do the candidates lay in 3ppm asper the machines dataset

4. How many molecules in matches are unkown to pubchem, filter!(look up with CIDs)

5. How many adducts in matches are not listed in pubchem, filter!

6. How many matched molecules are in the result's candidate list, filter!(look up identifier)

7. How many of the matched molecules, filtered prior are Ranked@1


### Variable: Loss 
- Filtering is needed to evalate the raw performance of JESTR
- However filtering is not realistic since it is not possible when no ground truth is given
- Tracks the limitation of this approach in the real world setting

In [ ]:
#---------------------------------
loss = 0 #tracks the percentage of matches that are lost due to filtering
#---------------------------------

### 1. Import datasets
- ground truth dataset 
- pubchem lite dataset
- spectra with assigned adduct 
- jestr result
- candidate bins

In [ ]:
import json 
import pandas as pd
import numpy as np
import re
import pickle
import matplotlib.pyplot as plt

In [ ]:
with open('../../data/medical/Inhousematch.csv') as f:
    matches = pd.read_csv(f)
print(f"Matches columns: {matches.columns}")

with open ('../../data/pubchemlite/PubChemLite_CCSbase_20260529.csv') as f:
    pubchem = pd.read_csv(f)
print(f"PubChem columns: {pubchem.columns}")


with open ('../../data/medical/RFA_MSMS_full_precursor.json') as f:
    rfa = json.load(f)
print(f"RFA keys: {rfa[0].keys()}")

with open("../../experiments/20260716_PUBCHEM_sample_run_3/result_RFA_MSMS_full_precursor.pkl", "rb") as f:
    result_medical = pickle.load(f)
    result_medical_df = pd.DataFrame(result_medical)
print(f"Result medical columns: {result_medical_df.columns}")

with open("../../data/pubchemlite/precursor_bins.json") as f:
    precursor_bins = json.load(f)


In [ ]:
mzmatches = matches[matches['Spectrum'].str.contains("m/z")]
print(f"Number of matches with m/z identifier: {len(mzmatches)}")

nmatches = matches[matches['Spectrum'].str.contains("n")]
print(f"Number of matches with n identifier: {len(nmatches)}")

adductsmz = set(mzmatches['Adduct'])
# adducts in n matches contain multiple adducts, we will split here by ","
adductsn = set(nmatches['Adduct'].str.split(',').explode())
adducts = adductsmz.union(adductsn)
print(f"Adducts in matches: {adducts}")

### Matches not in Results
- how many of the inhouse matches are part of the 9556 Spectra
- should be all

In [ ]:
# set of result identifiers
def package_identifier(identifier):
    # remove everything after "m/z" or "n", but keep the marker
    if "m/z" in identifier:
        part1 = identifier.split("m/z")[0].strip()
        return part1 + "m/z"
    elif "n" in identifier:
        part1 = identifier.split("n")[0].strip()
        return part1 + "n"
    else:
        return identifier
result_identifiers = set(result_medical_df['identifier'].apply(package_identifier))
not_in_results = set(matches['Spectrum']).difference(result_identifiers)
#result_identifiers
matches = matches[~matches['Spectrum'].isin(not_in_results)]

### 2. Molecules unkown to pubchem (Filter: "_mol")
- how many molecules of those in the matches are found in pubchem
- matching with the CID's

In [ ]:
# 1. create a dictionary for pubchemlite with the CID as key and the smiles ans MonoisotopicMass value
pubchem_dict = dict()
count = 0
for compound in pubchem.itertuples():
    monoisotopic_mass = compound.MonoisotopicMass
    smiles = compound.SMILES
    CIDs = compound.Related_CIDs.split()
    if len(CIDs) > 1:
        #print(f"Compound: {compound.CompoundName}, Related CIDs: {CIDs}")
        for CID in CIDs:
            pubchem_dict[CID] =  smiles, monoisotopic_mass
    else:
        pubchem_dict[CIDs[0]] =  smiles, monoisotopic_mass
# 2. filter the matches 
rows = list()
for match in matches.itertuples():
    CID = match.CID
    if CID in pubchem_dict:
        Spectrum = match.Spectrum
        Adduct = match.Adduct
        rows.append((Spectrum, Adduct, CID))
match_mol = pd.DataFrame(rows, columns=['Spectrum', 'Adduct', 'CID'])

print(f"Applied mol filter, from {len(matches)} matches, {len(match_mol)} matches report molecules listed in PubChemLite")
print(f"Percentage of matches lost due to mol filtering: {round((len(matches)-len(match_mol))/len(matches)*100, 2)}%")



### 3. Adducts unknown to pubchem (Filter: "_addu")
- how many matches contain adducts that are not listed in pubchem
- unpack n matches to be sepparate for each molecule

In [ ]:
# ADDUCT_MASS = {
#     "[M+H]+": 1.007276,
#     "[M+Na]+": 22.989218,
#     "[M+K]+": 38.963158,
#     "[M-H]-": -1.007276,
#     "[M+NH4]+": 18.033823,
#     "[M+H-H2O]+": 1.007276 - 18.010565,
#     "[M+HCOO]-": 45.997654,
#     "[M+CH3COO]-": 59.013851,
#     "[M+Na-2H]-": 22.989218 - 2*1.007825,
#     "[M]+": 0.0,
#     "[M]-": 0.0,
# }
#From UC-davis: https://fiehnlab.ucdavis.edu/staff/kind/Metabolomics/MS-Adduct-Calculator
ADDUCT_MASS = {
        "[M+H]+":       1.007276,
        "[M+Na]+":     22.989218,
        "[M-H]-":      -1.007276,
        "[M+NH4]+":    18.033823,
        "[M+K]+":      38.963158,
        "[M+H-H2O]+": -17.002740,
        "[M+HCOO]-":   44.998201,
        "[M+CH3COO]-": 59.013851,
        "[M+Na-2H]-":  20.974666,
        "[M]+":        -0.00054858,
        "[M]-":         0.00054858,
    }

rows = list()
for match in match_mol.itertuples():
    adductlist = match.Adduct.split(",")
    adduct_filtered = []
    for adduct in adductlist:
        if adduct in ADDUCT_MASS:
            adduct_filtered.append(adduct)
    if len(adduct_filtered) > 0:
        rows.append((match.Spectrum, ",".join(adduct_filtered), match.CID))

match_mol_addu = pd.DataFrame(rows, columns=['Spectrum', 'Adduct', 'CID'])
print(f"Applied adduct filter, from {len(match_mol)} matches, {len(match_mol_addu)} matches report adducts in the ADDUCT_MASS dictionary")
print(f"Percentage of matches lost due to adduct filtering: {round((len(match_mol)-len(match_mol_addu))/len(match_mol)*100, 2)}%")

In [ ]:
result_medical_df.head()

### 4. Matches in candidate set (Filter: "_cand")
- how many matches are in the candidate set
- use the pubchem_dict to match CIDs of matches to smiles

In [ ]:
#1. Use the precursor bin for candidate matching. Dictionary with centroid mass as key and candidates as value.

def pack_identifier(identifier):
    # remove everything after "m/z" or "n", but keep the marker
    if "m/z" in identifier:
        part1 = identifier.split("m/z")[0].strip()
        return part1 + "m/z"
    elif "n" in identifier:
        part1 = identifier.split("n")[0].strip()
        return part1 + "n"
    else:
        return identifier

# 3. create dictionary with identifier as key and a list of tuple with pubchemlite adducts and precursor masses
precursor_dict = dict()
for spectrum in rfa:
    identifier = pack_identifier(spectrum['identifier'])
    #if identifier ends with n take the adducts reported in RFA
    if identifier.endswith("n"):
        adduct = spectrum['adduct'].replace(" ", "")
        precursor = spectrum['precursor_mz']
        if identifier not in precursor_dict:
            precursor_dict[identifier] = [(adduct, precursor)]
        else:
            precursor_dict[identifier].extend([(adduct, precursor)])
    #if identifier ends with m/z get all adducts from pubchemlite (RFA adducts for mz are still sirius)
    if identifier.endswith("m/z"):
        adducts = ADDUCT_MASS.keys()
        precursor = spectrum['precursor_mz']
        for adduct in adducts:
            if identifier not in precursor_dict:
                precursor_dict[identifier] = [(adduct, precursor)]
            else:
                precursor_dict[identifier].extend([(adduct, precursor)])

print(f"Entry for identifier 1.13_131.0338m/z: {precursor_dict.get('1.13_131.0338m/z', [])}")
print(f"Entry for identifier 15.83_278.2235n: {precursor_dict.get('15.83_278.2235n', [])}")


In [ ]:
# Collect all candidates for each identifier
candidates_collected = dict()
for result in result_medical_df.itertuples():
    identifier = pack_identifier(result.identifier)
    candidates_bin = precursor_bins[result.bin]   # list of SMILES for this bin
    bin_id = result.bin                           # centroid / bin index

    if identifier not in candidates_collected:
        candidates_collected[identifier] = {
            "smiles": set(candidates_bin),
            "bins": {bin_id},
        }
    else:
        candidates_collected[identifier]["smiles"].update(candidates_bin)
        candidates_collected[identifier]["bins"].add(bin_id)
print(f"Candidates for 15.56_370.2708m/z: {candidates_collected['15.56_370.2708m/z']}")

# 2. Check matches against candidates dictionary
# check how close the centroid precursor mass is to all possible pubchem lite precursors of that match
def get_closest_precursor_ppm(neutralmass, centroid_mass):
    adduct_masses = np.array(list(ADDUCT_MASS.values()))
    precursors = neutralmass + adduct_masses
    closest_idx = np.argmin(np.abs(precursors - centroid_mass))
    ppm_error = (centroid_mass - precursors[closest_idx]) / precursors[closest_idx] * 1e6
    return precursors[closest_idx], ppm_error

def get_ppm_error(monoisotopicmass1, adduct1, precursormass2):
    if adduct1 not in ADDUCT_MASS:
        print(f"Adduct {adduct1} not found in ADDUCT_MASS dictionary")
    else:
        precursormass1 = float(monoisotopicmass1) + ADDUCT_MASS[adduct1]
        ppm_error = (precursormass1 - precursormass2) / precursormass2 * 1e6
        return ppm_error, precursormass1
    

def get_precursor(identifier, adduct):
    if identifier in precursor_dict:
        for adduct_precursor in precursor_dict[identifier]:
            if adduct_precursor[0] == adduct:
                return adduct_precursor[1]
    else:
        print(f"Identifier {identifier} not found in precursor dictionary")
    

rows = list()
not_found = list()
for match in match_mol_addu.itertuples():
    # 1. Get the smiles
    CID = match.CID
    #print(f" pubchemdict entry for CID {CID}: {pubchem_dict[CID]}")
    smiles, monoisotopic_mass = pubchem_dict[CID]
    # 2. Check if the match is in the candidates dictionary
    identifier = match.Spectrum
    if identifier in candidates_collected:
        candidate_smiles = candidates_collected[identifier]["smiles"]
        bin_idx = candidates_collected[identifier]["bins"]
        # if smiles =="CCCCC/C=C\\C/C=C\\C/C=C\\C/C=C\\CCCC(=O)OCCN":
        #     print(f"Candidate smiles for {identifier}: binidx: {bin_idx}, smiles: {candidate_smiles}")
        if smiles in candidate_smiles:
            rows.append((match.Spectrum, match.Adduct, match.CID, smiles, monoisotopic_mass))
        else:
            #precursor, ppm_error = get_closest_precursor_ppm(monoisotopic_mass, bin_idx)
            #print(f"SMILES: {smiles} not found in candidates for identifier: {identifier}")
            precursor = get_precursor(identifier, match.Adduct)
            if precursor is None:
                print(f"No precursor found for identifier: {identifier} and adduct: {match.Adduct}")
                continue
            ppm_error, precursor1 = get_ppm_error(monoisotopic_mass, match.Adduct, precursor)
            not_found.append((match.Spectrum, match.Adduct, match.CID, smiles, precursor, precursor1, ppm_error, bin_idx))
    else:
        raise AttributeError(f"Identifier {identifier} not found in candidates dictionary")
match_mol_addu_cand = pd.DataFrame(rows, columns=['Spectrum', 'Adduct', 'CID', 'SMILES', 'MonoisotopicMass'])
print(f"Applied candidate filter, from {len(match_mol_addu)} matches, {len(rows)} matches report candidates in the candidates dictionary")
print(f"Percentage of matches lost due to candidate filtering: {round((len(match_mol_addu)-len(rows))/len(match_mol_addu)*100, 2)}%")
print(f"Matches not found in candidates dictionary: {not_found}")
print(f"Match not found with minimal ppm error: {min([match for match in not_found if match[6] < 3], key=lambda x: x[6]) if [match for match in not_found if match[6] < 3] else None}")
print(f"Match found with maximal ppm error: {max([match for match in not_found if match[6] < 3], key=lambda x: x[6]) if [match for match in not_found if match[6] < 3] else None}")

the precursor masses of all matches not in candidates have an error larger than 2ppm from the real precursor mass, we decide to not use a sliding window algo since this will be an even larger index with 3.8 000 000 Indices. 100% coverage under 2ppm is reasonable.

### 5. Final Ranking 

In [ ]:
result_scores = dict()
for result in result_medical_df.itertuples():
    identifier = pack_identifier(result.identifier)
    candidate_smiles = result.candidates
    scores = result.scores
    if identifier in result_scores:
        result_scores[identifier]["candidates"].extend(candidate_smiles)
        result_scores[identifier]["scores"].extend(scores)
    else:
        result_scores[identifier] = {
            "candidates": candidate_smiles,
            "scores": scores
        }

def get_rank(candidates, scores, target_smiles):
    """Target smiles has to be in candidates"""
    # sort candidates by scores in descending order
    sorted_candidates = [x for _, x in sorted(zip(scores, candidates), reverse=True)]
    rank = sorted_candidates.index(target_smiles) + 1
    score = scores[candidates.index(target_smiles)]
    return rank, score

ranks= []
total_matches_threshold = []
for match in match_mol_addu_cand.itertuples():
    smiles = match.SMILES
    if match.Spectrum in result_scores:
        candidates = result_scores[match.Spectrum]["candidates"]
        scores = result_scores[match.Spectrum]["scores"]
        #if first ranked molecule is below 0.7, we will not count it as a match
        
        if smiles in candidates:
            if max(scores) < 0.7:
                continue
            total_matches_threshold.append(smiles)
            rank, score = get_rank(candidates, scores, smiles)
            ranks.append(rank)
            #print(f"Match {smiles} found in candidates for {match.Spectrum} with rank: {rank}")
        else:
            if max(scores) < 0.7:
                continue
            total_matches_threshold.append(smiles)

            #print(f"Match {smiles} not found in candidates for {match.Spectrum}")
print(f"Total matches with ranks: {len(ranks)},  {len(match_mol_addu_cand) - len(ranks)} not under top 20")
at1 = len([r for r in ranks if r == 1]) 
at5 = len([r for r in ranks if r <= 5]) 
at20 = len([r for r in ranks if r <= 20])
print(f"Recall @1: {at1 / len(total_matches_threshold):.2%} denovo : {at1 / len(matches):.2%}")
print(f"Recall @5: {at5 / len(total_matches_threshold):.2%} denovo : {at5 / len(matches):.2%}")
print(f"Recall @20: {at20 / len(total_matches_threshold):.2%} denovo : {at20 / len(matches):.2%}")
print(f" at1: {at1}, at5: {at5}, at20: {at20}, total matches with ranks: {len(ranks)}, total matches: {len(total_matches_threshold)}")


In [ ]:
scores_at1_target = []     # @1 is the true target
scores_at1_not_target = [] # @1 is NOT the true target

for match in match_mol_addu_cand.itertuples():
    smiles = match.SMILES
    spectrum = match.Spectrum

    if spectrum not in result_scores:
        continue

    candidates = result_scores[spectrum]["candidates"]
    scores = result_scores[spectrum]["scores"]

    # sort candidates by score descending
    sorted_pairs = sorted(zip(scores, candidates), reverse=True)
    sorted_scores = [s for s, c in sorted_pairs]
    sorted_candidates = [c for s, c in sorted_pairs]

    # score of the top-1 candidate
    score_at1 = sorted_scores[0]
    top1_candidate = sorted_candidates[0]

    # check if @1 is the true target
    if top1_candidate == smiles:
        scores_at1_target.append(score_at1)
    else:
        scores_at1_not_target.append(score_at1)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.hist(scores_at1_target, bins=40, alpha=0.6, label='@1 is TRUE target', color='green')
plt.hist(scores_at1_not_target, bins=40, alpha=0.6, label='@1 is NOT target', color='red')

plt.title("Confidence Distribution of @1 Candidate")
plt.xlabel("Raw Score (Cosine Similarity)")
plt.ylabel("Count")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
gap_correct = []   # @1 is true target
gap_wrong = []     # @1 is NOT true target

for match in match_mol_addu_cand.itertuples():
    smiles = match.SMILES
    spectrum = match.Spectrum

    if spectrum not in result_scores:
        continue

    candidates = result_scores[spectrum]["candidates"]
    scores = result_scores[spectrum]["scores"]

    if smiles not in candidates:
        continue

    # sort candidates by score descending
    sorted_pairs = sorted(zip(scores, candidates), reverse=True)
    sorted_scores = [s for s, c in sorted_pairs]
    sorted_candidates = [c for s, c in sorted_pairs]

    # rank of true molecule
    rank = sorted_candidates.index(smiles) + 1

    # score gap between @1 and @2
    if len(sorted_scores) >= 2:
        gap = sorted_scores[0] - sorted_scores[1]
    else:
        gap = None  # rare case: only one candidate

    # group assignment
    if rank == 1:
        gap_correct.append(gap)
    else:
        gap_wrong.append(gap)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.hist(gap_correct, bins=40, alpha=0.6, label='@1 is TRUE target', color='green')
plt.hist(gap_wrong, bins=40, alpha=0.6, label='@1 is NOT target', color='red')

plt.title("Score Gap Distribution: score(@1) - score(@2)")
plt.xlabel("Score Gap")
plt.ylabel("Count")
plt.legend()
plt.grid(True)
plt.show()